In [0]:
# DAC043 CT chest/abdomen/pelvis report extract
# Drop-in project notebook: move to /Workspace/Shared/ADC-DB/Projects and Run All.

In [0]:
# Configuration
project_identifier = "dac043"
TARGET_CATALOG = "5_projects"
TARGET = f"{TARGET_CATALOG}.{project_identifier}"
PSEUDONYM_SALT = "5055d702c374f9dfd3e48cac740797bdb9d2ce7c94994be6ec601a6e6006cd6b"

MIN_EXAM_DATE = "2016-01-01"
TARGET_N = 25000
SAMPLE_SEED = "DAC043-2026-09-04"
PIPELINE_VERSION = "dac043.v1.1"

CAP_EVENT_CDS = {
    6183849: ("CT Thorax & abdo & pelvis with contrast", "CAP", True),
    6184208: ("CT Thorax & abdomen & pelvis", "CAP", False),
    6182768: ("CT Neck & thorax & abdo & pelvis & cont", "NECK_CAP", True),
    6183516: ("CT Neck & thorax & abdomen & pelvis", "NECK_CAP", False),
    6182338: ("CT Thorax without Cont & Abdo with Cont", "CA", True),
}
EVENT_CD_SQL = ",".join(str(code) for code in sorted(CAP_EVENT_CDS))
EXAM_FAMILY_CASE = " ".join(
    f"WHEN {code} THEN '{values[1]}'"
    for code, values in CAP_EVENT_CDS.items()
)
CONTRAST_CASE = " ".join(
    f"WHEN {code} THEN {str(values[2]).lower()}"
    for code, values in CAP_EVENT_CDS.items()
)

print(f"Project target: {TARGET}")
print(f"CAP event codes: {EVENT_CD_SQL}")
print(f"Sample: {TARGET_N} rows, seed {SAMPLE_SEED}, exams from {MIN_EXAM_DATE}")

In [0]:
# Project schema and cleanup.
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {TARGET}
COMMENT 'DAC043 CT chest/abdomen/pelvis report extract'
""")

managed_names = {
    "ct_report_tre",
    "ct_report_qmul",
    "turnaround_band_evidence",
    "anon_leakage_qa",
    "release_checks",
    "schema",
    "_pseudonym_config",
}
existing_objects = {
    row["table_name"]: row["table_type"]
    for row in spark.sql(f"""
      SELECT table_name, table_type
      FROM {TARGET_CATALOG}.information_schema.tables
      WHERE table_schema = '{project_identifier}'
    """).collect()
    if row["table_name"] in managed_names
}
for object_name, object_type in existing_objects.items():
    if object_type == "VIEW":
        spark.sql(f"DROP VIEW IF EXISTS {TARGET}.{object_name}")
    else:
        spark.sql(f"DROP TABLE IF EXISTS {TARGET}.{object_name}")
    print(f"Dropped old object: {TARGET}.{object_name}")

In [0]:
# Full TRE view.
spark.sql(f"""
CREATE TABLE {TARGET}.ct_report_tre AS
WITH blob_ranked AS (
  SELECT
    EVENT_ID,
    EVENT_CD,
    EVENT_CD_DESC,
    BLOB_TEXT,
    anon_text,
    TEXT_LENGTH,
    VERIFIED_PRSNL_ID,
    VERIFIED_DT_TM,
    PERFORMED_PRSNL_ID,
    ENCODING,
    parser_version,
    anon_status,
    anon_redactor_version,
    anon_redaction_count,
    STATUS,
    ROW_NUMBER() OVER (
      PARTITION BY EVENT_ID
      ORDER BY
        CASE WHEN BLOB_TEXT IS NOT NULL AND length(trim(BLOB_TEXT)) > 0 THEN 0 ELSE 1 END,
        CASE WHEN anon_text IS NOT NULL AND length(trim(anon_text)) > 0 THEN 0 ELSE 1 END,
        CASE WHEN VALID_UNTIL_DT_TM > current_timestamp() THEN 0 ELSE 1 END,
        VALID_FROM_DT_TM DESC NULLS LAST,
        UPDT_CNT DESC NULLS LAST,
        parser_version DESC NULLS LAST,
        TEXT_LENGTH DESC NULLS LAST,
        BLOB_VERSION_ID DESC NULLS LAST
    ) AS rn
  FROM 4_prod.bronze.mill_blob_text
  WHERE EVENT_CD IN ({EVENT_CD_SQL})
),
blob AS (
  SELECT *
  FROM blob_ranked
  WHERE rn = 1
),
radiology_event AS (
  SELECT
    EVENT_ID,
    PERSON_ID,
    ENCNTR_ID,
    REFERENCE_NBR,
    LEFT(REFERENCE_NBR, 16) AS ACCESSION_NBR,
    EXAM_TYPE_CODE,
    COALESCE(EVENT_END_DT_TM_CLEAN, EVENT_START_DT_TM_CLEAN) AS MILL_EVENT_DT_TM,
    IN_ERROR_IND
  FROM 4_prod.bronze.map_radiology_event
  WHERE EVENT_CD IN ({EVENT_CD_SQL})
),
base AS (
  SELECT
    e.EVENT_ID,
    e.PERSON_ID,
    e.ENCNTR_ID,
    e.REFERENCE_NBR,
    e.ACCESSION_NBR,
    e.EXAM_TYPE_CODE,
    e.MILL_EVENT_DT_TM,
    e.IN_ERROR_IND,
    b.EVENT_CD,
    b.EVENT_CD_DESC,
    b.BLOB_TEXT,
    b.anon_text,
    b.TEXT_LENGTH,
    b.VERIFIED_PRSNL_ID,
    b.VERIFIED_DT_TM,
    b.PERFORMED_PRSNL_ID,
    b.ENCODING,
    b.parser_version,
    b.anon_status,
    b.anon_redactor_version,
    b.anon_redaction_count,
    b.STATUS
  FROM radiology_event e
  INNER JOIN blob b
    ON b.EVENT_ID = e.EVENT_ID
),
pacs_ranked AS (
  SELECT
    b.*,
    p.PACS_EXAMINATION_ID,
    p.STUDY_INSTANCE_UID,
    p.EXAMINATION_DT_TM AS PACS_EXAM_DT_TM,
    p.EXAMINATION_CODE AS PACS_EXAMINATION_CODE,
    p.EXAMINATION_DESCRIPTION,
    p.MODALITY,
    p.BODY_PART,
    p.INSTITUTION,
    p.CLINICAL_QUESTION,
    p.CLINICAL_ANAMNESIS,
    p.REFERRING_UNIT,
    p.IMAGE_COUNT,
    p.SERIES_COUNT_MEASURED,
    p.STORED_SIZE_BYTES,
    p.NICIP_SNOMED_CODE,
    ROW_NUMBER() OVER (
      PARTITION BY b.EVENT_ID
      ORDER BY
        CASE WHEN upper(trim(p.EXAMINATION_CODE)) = upper(trim(b.EXAM_TYPE_CODE)) THEN 0 ELSE 1 END,
        CASE WHEN p.STUDY_INSTANCE_UID IS NOT NULL THEN 0 ELSE 1 END,
        abs(datediff(p.EXAMINATION_DT_TM, b.MILL_EVENT_DT_TM)) ASC NULLS LAST,
        p.IMAGE_COUNT DESC NULLS LAST,
        p.PACS_EXAMINATION_ID DESC NULLS LAST
    ) AS pacs_rn
  FROM base b
  LEFT JOIN 4_prod.bronze.map_pacs_examination p
    ON p.REQUEST_ID_STRING = b.ACCESSION_NBR
),
eligible AS (
  SELECT
    *,
    COALESCE(PACS_EXAM_DT_TM, MILL_EVENT_DT_TM) AS EXAMINATION_DT_TM,
    substr(
      sha2(concat('{SAMPLE_SEED}', ':', CAST(EVENT_ID AS STRING)), 256),
      1,
      16
    ) AS SAMPLE_RANK
  FROM pacs_ranked
  WHERE pacs_rn = 1
    AND COALESCE(PACS_EXAM_DT_TM, MILL_EVENT_DT_TM) >= TIMESTAMP'{MIN_EXAM_DATE}'
    AND COALESCE(PACS_EXAM_DT_TM, MILL_EVENT_DT_TM) < current_timestamp()
    AND BLOB_TEXT IS NOT NULL
    AND length(trim(BLOB_TEXT)) >= 100
    AND anon_text IS NOT NULL
    AND length(trim(anon_text)) >= 50
    AND anon_status = 'anonymized'
    AND STATUS = 'Decoded'
    AND COALESCE(IN_ERROR_IND, false) = false
    AND rlike(ACCESSION_NBR, '^UK[A-Z]{{3}}[0-9]{{11}}$')
),
accession_unique AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY ACCESSION_NBR
      ORDER BY SAMPLE_RANK ASC, EVENT_ID ASC
    ) AS accession_rn
  FROM eligible
),
cohort_ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      ORDER BY SAMPLE_RANK ASC, EVENT_ID ASC
    ) AS cohort_rn
  FROM accession_unique
  WHERE accession_rn = 1
),
cohort AS (
  SELECT *
  FROM cohort_ranked
  WHERE cohort_rn <= {TARGET_N}
),
rx_ranked AS (
  SELECT
    c.EVENT_ID,
    x.ExaminationStationName,
    x.ExaminationInstitution,
    x.ExaminationScheduledDate,
    x.ExaminationStat,
    ROW_NUMBER() OVER (
      PARTITION BY c.EVENT_ID
      ORDER BY x.ADC_UPDT DESC NULLS LAST, x.ExaminationId DESC NULLS LAST
    ) AS rn
  FROM cohort c
  INNER JOIN 4_prod.raw.pacs_examinations x
    ON x.ExaminationId = c.PACS_EXAMINATION_ID
),
rx AS (
  SELECT *
  FROM rx_ranked
  WHERE rn = 1
),
request_ranked AS (
  SELECT
    c.EVENT_ID,
    r.RequestReferringPhysician,
    r.RequestReferringUnit,
    r.RequestQuestion,
    r.RequestAnamnesis,
    ROW_NUMBER() OVER (
      PARTITION BY c.EVENT_ID
      ORDER BY r.ADC_UPDT DESC NULLS LAST, r.RequestId DESC NULLS LAST
    ) AS rn
  FROM cohort c
  INNER JOIN 4_prod.raw.pacs_requests r
    ON r.RequestIdString = c.ACCESSION_NBR
),
request AS (
  SELECT *
  FROM request_ranked
  WHERE rn = 1
),
encounter_ranked AS (
  SELECT
    c.EVENT_ID,
    e.encntr_type_class_desc,
    e.encntr_type_desc,
    e.med_service_desc,
    ROW_NUMBER() OVER (
      PARTITION BY c.EVENT_ID
      ORDER BY
        CASE WHEN e.encntr_type_class_desc IS NOT NULL THEN 0 ELSE 1 END,
        e.ADC_UPDT DESC NULLS LAST
    ) AS rn
  FROM cohort c
  INNER JOIN 4_prod.bronze.map_encounter e
    ON e.ENCNTR_ID = c.ENCNTR_ID
),
encounter AS (
  SELECT *
  FROM encounter_ranked
  WHERE rn = 1
),
identifier_ranked AS (
  SELECT
    i.PERSON_ID,
    i.ALIAS_TYPE,
    i.ALIAS_VALUE,
    ROW_NUMBER() OVER (
      PARTITION BY i.PERSON_ID, i.ALIAS_TYPE
      ORDER BY
        CASE WHEN i.ALIAS_VALUE IS NOT NULL AND length(trim(i.ALIAS_VALUE)) > 0 THEN 0 ELSE 1 END,
        CASE WHEN i.CURRENT_IND THEN 0 ELSE 1 END,
        CASE WHEN i.ACTIVE_IND = 1 THEN 0 ELSE 1 END,
        CASE
          WHEN i.END_EFFECTIVE_DT_TM_CLEAN IS NULL
            OR i.END_EFFECTIVE_DT_TM_CLEAN > current_timestamp() THEN 0
          ELSE 1
        END,
        i.BEG_EFFECTIVE_DT_TM_CLEAN DESC NULLS LAST,
        i.UPDT_CNT DESC NULLS LAST,
        i.ALIAS_VALUE ASC NULLS LAST
    ) AS rn
  FROM 4_prod.bronze.map_patient_identifier i
  INNER JOIN (
    SELECT DISTINCT PERSON_ID
    FROM cohort
  ) c
    ON c.PERSON_ID = i.PERSON_ID
  WHERE i.ALIAS_TYPE IN ('MRN', 'NHS')
),
identifiers AS (
  SELECT
    PERSON_ID,
    MAX(CASE WHEN ALIAS_TYPE = 'MRN' THEN ALIAS_VALUE END) AS MRN,
    MAX(CASE WHEN ALIAS_TYPE = 'NHS' THEN ALIAS_VALUE END) AS NHS_NUMBER
  FROM identifier_ranked
  WHERE rn = 1
  GROUP BY PERSON_ID
),
accession_order AS (
  SELECT
    LEFT(r.REFERENCE_NBR, 16) AS ACCESSION_NBR,
    MAX(r.ORDER_ID) AS ORDER_ID,
    COUNT(DISTINCT r.ORDER_ID) AS ORDER_ID_CANDIDATES
  FROM 4_prod.bronze.map_radiology_event r
  WHERE r.EVENT_CD IN ({EVENT_CD_SQL})
    AND r.ORDER_ID IS NOT NULL
    AND rlike(LEFT(r.REFERENCE_NBR, 16), '^UK[A-Z]{{3}}[0-9]{{11}}$')
  GROUP BY LEFT(r.REFERENCE_NBR, 16)
),
assembled AS (
  SELECT
    substr(
      sha2(concat('{PSEUDONYM_SALT}', ':ACC:', c.ACCESSION_NBR), 256),
      1,
      32
    ) AS PSEUDO_ID,
    substr(
      sha2(concat('{PSEUDONYM_SALT}', ':PER:', CAST(c.PERSON_ID AS STRING)), 256),
      1,
      32
    ) AS PSEUDO_PATIENT_ID,
    c.ACCESSION_NBR,
    c.STUDY_INSTANCE_UID,
    c.EVENT_ID,
    c.PERSON_ID,
    c.ENCNTR_ID,
    c.PACS_EXAMINATION_ID,
    ids.MRN,
    ids.NHS_NUMBER,
    c.EXAMINATION_DT_TM,
    c.BLOB_TEXT AS REPORT_TEXT,
    c.anon_text AS REPORT_TEXT_ANON,
    c.anon_status AS ANON_STATUS,
    c.anon_redactor_version AS ANON_REDACTOR_VERSION,
    c.anon_redaction_count AS ANON_REDACTION_COUNT,
    c.EVENT_CD_DESC AS EXAMINATION_NAME,
    c.EXAMINATION_DESCRIPTION AS EXAMINATION_DESCRIPTION_PACS,
    COALESCE(c.EXAM_TYPE_CODE, c.PACS_EXAMINATION_CODE) AS EXAMINATION_CODE,
    person.gender_display AS SEX,
    LEAST(
      CAST(floor(datediff(c.EXAMINATION_DT_TM, person.birth_date) / 365.25) AS INT),
      90
    ) AS AGE_AT_EXAM,
    COALESCE(c.CLINICAL_ANAMNESIS, request.RequestAnamnesis) AS CLINICAL_DETAILS,
    COALESCE(c.CLINICAL_QUESTION, request.RequestQuestion) AS REASON_FOR_REFERRAL,
    request.RequestReferringPhysician AS REFERRING_CLINICIAN,
    COALESCE(rx.ExaminationInstitution, c.INSTITUTION) AS PERFORMING_SITE,
    COALESCE(request.RequestReferringUnit, c.REFERRING_UNIT) AS DEPARTMENT,
    encounter.med_service_desc AS MED_SERVICE,
    rx.ExaminationStationName AS ROOM,
    c.VERIFIED_DT_TM AS AUTHORISATION_DT_TM,
    orders.ORIG_ORDER_DT_TM_CLEAN AS REQUEST_DT_TM,
    CASE
      WHEN orders.ORIG_ORDER_DT_TM_CLEAN IS NOT NULL THEN 'MILL_ORDER_VIA_ACCESSION'
      ELSE 'UNAVAILABLE'
    END AS REQUEST_DT_SOURCE,
    rx.ExaminationStat AS PRIORITY_CD,
    rx.ExaminationScheduledDate AS APPOINTMENT_DT_TM,
    encounter.encntr_type_class_desc AS REQUEST_CATEGORY,
    encounter.encntr_type_desc AS ENCOUNTER_TYPE,
    verifier.NAME_FULL_FORMATTED AS AUTHORISING_RADIOLOGIST,
    c.VERIFIED_PRSNL_ID AS AUTHORISING_RADIOLOGIST_ID,
    performer.NAME_FULL_FORMATTED AS PERFORMED_RADIOGRAPHER,
    'CAP' AS EXAM_FAMILY_SET,
    CASE c.EVENT_CD {EXAM_FAMILY_CASE} END AS EXAM_FAMILY,
    CASE c.EVENT_CD {CONTRAST_CASE} END AS CONTRAST_IND,
    c.MODALITY,
    c.BODY_PART,
    c.NICIP_SNOMED_CODE,
    c.IMAGE_COUNT,
    c.SERIES_COUNT_MEASURED,
    c.STORED_SIZE_BYTES,
    length(c.BLOB_TEXT) AS REPORT_CHAR_LENGTH,
    length(c.anon_text) AS ANON_REPORT_CHAR_LENGTH,
    CASE
      WHEN c.ENCODING IN ('ISO-8859-1', 'Windows-1252', 'latin-1', 'cp775')
        AND c.parser_version IS NULL THEN 'LEGACY_SINGLE_BYTE_RTF_RISK'
      WHEN c.ENCODING = 'ascii' THEN 'OK_ASCII'
      ELSE 'OK'
    END AS TEXT_QUALITY_FLAG,
    person.deceased_display AS PATIENT_DECEASED,
    CASE
      WHEN person.deceased_dt_tm IS NOT NULL
        THEN datediff(person.deceased_dt_tm, c.EXAMINATION_DT_TM)
    END AS DAYS_EXAM_TO_DEATH,
    ROW_NUMBER() OVER (
      PARTITION BY c.PERSON_ID
      ORDER BY c.EXAMINATION_DT_TM ASC, c.EVENT_ID ASC
    ) AS SCAN_SEQ_FOR_PATIENT,
    datediff(
      c.EXAMINATION_DT_TM,
      LAG(c.EXAMINATION_DT_TM) OVER (
        PARTITION BY c.PERSON_ID
        ORDER BY c.EXAMINATION_DT_TM ASC, c.EVENT_ID ASC
      )
    ) AS DAYS_SINCE_PREV_CAP_SCAN,
    '{PIPELINE_VERSION}' AS PIPELINE_VERSION
  FROM cohort c
  LEFT JOIN rx
    ON rx.EVENT_ID = c.EVENT_ID
  LEFT JOIN request
    ON request.EVENT_ID = c.EVENT_ID
  LEFT JOIN encounter
    ON encounter.EVENT_ID = c.EVENT_ID
  LEFT JOIN identifiers ids
    ON ids.PERSON_ID = c.PERSON_ID
  LEFT JOIN 4_prod.bronze.map_person person
    ON person.person_id = c.PERSON_ID
  LEFT JOIN 4_prod.bronze.map_medical_personnel verifier
    ON verifier.PERSON_ID = c.VERIFIED_PRSNL_ID
  LEFT JOIN 4_prod.bronze.map_medical_personnel performer
    ON performer.PERSON_ID = c.PERFORMED_PRSNL_ID
  LEFT JOIN accession_order accession_order
    ON accession_order.ACCESSION_NBR = c.ACCESSION_NBR
  LEFT JOIN 4_prod.bronze.map_orders orders
    ON orders.ORDER_ID = accession_order.ORDER_ID
),
priority_stats AS (
  SELECT
    PRIORITY_CD,
    COUNT(*) AS n,
    percentile_approx(
      CAST(AUTHORISATION_DT_TM AS DOUBLE) - CAST(EXAMINATION_DT_TM AS DOUBLE),
      0.5
    ) / 3600.0 AS median_turnaround_hours
  FROM assembled
  WHERE AUTHORISATION_DT_TM IS NOT NULL
    AND AUTHORISATION_DT_TM > EXAMINATION_DT_TM
  GROUP BY PRIORITY_CD
)
SELECT
  a.*,
  CASE
    WHEN a.PRIORITY_CD IS NULL OR p.n IS NULL THEN 'UNRECORDED'
    WHEN p.median_turnaround_hours <= 6 THEN 'FAST'
    WHEN p.median_turnaround_hours <= 48 THEN 'MEDIUM'
    ELSE 'SLOW'
  END AS TURNAROUND_BAND
FROM assembled a
LEFT JOIN priority_stats p
  ON a.PRIORITY_CD <=> p.PRIORITY_CD
""")

spark.sql(f"""
COMMENT ON TABLE {TARGET}.ct_report_tre IS
'DAC043 full identifiable CT report extract. One row per sampled accession. Keep within the TRE.'
""")

In [0]:
# Queen Mary candidate view: approved anon text unchanged, no direct identifier
# columns and no DATE/TIMESTAMP columns.
spark.sql(f"""
CREATE TABLE {TARGET}.ct_report_qmul AS
SELECT
  PSEUDO_ID,
  PSEUDO_PATIENT_ID,
  REPORT_TEXT_ANON AS ANON_REPORT_TEXT,
  ANON_STATUS,
  ANON_REDACTOR_VERSION,
  ANON_REDACTION_COUNT,
  ANON_REPORT_CHAR_LENGTH AS REPORT_CHAR_LENGTH,
  EXAMINATION_NAME,
  EXAMINATION_CODE,
  EXAM_FAMILY,
  CONTRAST_IND,
  MODALITY,
  BODY_PART,
  NICIP_SNOMED_CODE,
  AGE_AT_EXAM,
  SEX,
  REQUEST_CATEGORY,
  PRIORITY_CD,
  TURNAROUND_BAND,
  YEAR(EXAMINATION_DT_TM) AS EXAM_YEAR,
  datediff(EXAMINATION_DT_TM, REQUEST_DT_TM) AS DAYS_REQUEST_TO_EXAM,
  datediff(AUTHORISATION_DT_TM, EXAMINATION_DT_TM) AS DAYS_EXAM_TO_AUTHORISATION,
  datediff(EXAMINATION_DT_TM, APPOINTMENT_DT_TM) AS DAYS_APPOINTMENT_TO_EXAM,
  SCAN_SEQ_FOR_PATIENT,
  DAYS_SINCE_PREV_CAP_SCAN,
  TEXT_QUALITY_FLAG,
  PIPELINE_VERSION
FROM {TARGET}.ct_report_tre
""")

spark.sql(f"""
COMMENT ON TABLE {TARGET}.ct_report_qmul IS
'DAC043 Queen Mary candidate view. Approved mill_blob_text.anon_text is used unchanged. No direct identifier columns or absolute structured dates. IG sign-off is still required.'
""")

In [0]:
# Tag both materialised outputs.
IG_TRE = {
    "PSEUDO_ID": ("0", "1"), "PSEUDO_PATIENT_ID": ("0", "1"),
    "ACCESSION_NBR": ("4", "2"), "STUDY_INSTANCE_UID": ("0", "1"),
    "EVENT_ID": ("0", "0"), "PERSON_ID": ("4", "2"), "ENCNTR_ID": ("4", "2"),
    "PACS_EXAMINATION_ID": ("0", "0"), "MRN": ("4", "2"), "NHS_NUMBER": ("4", "2"),
    "EXAMINATION_DT_TM": ("1", "1"), "REPORT_TEXT": ("4", "3"),
    "REPORT_TEXT_ANON": ("3", "2"), "ANON_STATUS": ("0", "0"),
    "ANON_REDACTOR_VERSION": ("0", "0"), "ANON_REDACTION_COUNT": ("0", "0"),
    "EXAMINATION_NAME": ("0", "0"), "EXAMINATION_DESCRIPTION_PACS": ("0", "0"),
    "EXAMINATION_CODE": ("0", "0"), "SEX": ("1", "1"), "AGE_AT_EXAM": ("1", "1"),
    "CLINICAL_DETAILS": ("3", "2"), "REASON_FOR_REFERRAL": ("3", "2"),
    "REFERRING_CLINICIAN": ("4", "3"), "PERFORMING_SITE": ("0", "0"),
    "DEPARTMENT": ("1", "0"), "MED_SERVICE": ("0", "0"), "ROOM": ("0", "0"),
    "AUTHORISATION_DT_TM": ("1", "1"), "REQUEST_DT_TM": ("1", "1"),
    "REQUEST_DT_SOURCE": ("0", "0"), "PRIORITY_CD": ("0", "0"),
    "APPOINTMENT_DT_TM": ("1", "1"), "REQUEST_CATEGORY": ("0", "0"),
    "ENCOUNTER_TYPE": ("0", "0"), "AUTHORISING_RADIOLOGIST": ("4", "3"),
    "AUTHORISING_RADIOLOGIST_ID": ("1", "1"), "PERFORMED_RADIOGRAPHER": ("4", "3"),
    "EXAM_FAMILY_SET": ("0", "0"), "EXAM_FAMILY": ("0", "0"),
    "CONTRAST_IND": ("0", "0"), "MODALITY": ("0", "0"), "BODY_PART": ("0", "0"),
    "NICIP_SNOMED_CODE": ("0", "0"), "IMAGE_COUNT": ("0", "0"),
    "SERIES_COUNT_MEASURED": ("0", "0"), "STORED_SIZE_BYTES": ("0", "0"),
    "REPORT_CHAR_LENGTH": ("0", "0"), "ANON_REPORT_CHAR_LENGTH": ("0", "0"),
    "TEXT_QUALITY_FLAG": ("0", "0"), "PATIENT_DECEASED": ("1", "2"),
    "DAYS_EXAM_TO_DEATH": ("1", "2"), "SCAN_SEQ_FOR_PATIENT": ("0", "0"),
    "DAYS_SINCE_PREV_CAP_SCAN": ("0", "0"), "PIPELINE_VERSION": ("0", "0"),
    "TURNAROUND_BAND": ("0", "0"),
}

IG_QMUL = {
    "PSEUDO_ID": ("0", "1"), "PSEUDO_PATIENT_ID": ("0", "1"),
    "ANON_REPORT_TEXT": ("3", "2"), "ANON_STATUS": ("0", "0"),
    "ANON_REDACTOR_VERSION": ("0", "0"), "ANON_REDACTION_COUNT": ("0", "0"),
    "REPORT_CHAR_LENGTH": ("0", "0"), "EXAMINATION_NAME": ("0", "0"),
    "EXAMINATION_CODE": ("0", "0"), "EXAM_FAMILY": ("0", "0"),
    "CONTRAST_IND": ("0", "0"), "MODALITY": ("0", "0"), "BODY_PART": ("0", "0"),
    "NICIP_SNOMED_CODE": ("0", "0"), "AGE_AT_EXAM": ("1", "1"),
    "SEX": ("1", "1"), "REQUEST_CATEGORY": ("0", "0"), "PRIORITY_CD": ("0", "0"),
    "TURNAROUND_BAND": ("0", "0"), "EXAM_YEAR": ("1", "0"),
    "DAYS_REQUEST_TO_EXAM": ("0", "0"), "DAYS_EXAM_TO_AUTHORISATION": ("0", "0"),
    "DAYS_APPOINTMENT_TO_EXAM": ("0", "0"), "SCAN_SEQ_FOR_PATIENT": ("0", "0"),
    "DAYS_SINCE_PREV_CAP_SCAN": ("0", "0"), "TEXT_QUALITY_FLAG": ("0", "0"),
    "PIPELINE_VERSION": ("0", "0"),
}

for table_name, tags in (("ct_report_tre", IG_TRE), ("ct_report_qmul", IG_QMUL)):
    columns = [field.name for field in spark.table(f"{TARGET}.{table_name}").schema]
    missing = sorted(set(columns) - set(tags))
    assert not missing, f"Missing IG classifications for {table_name}: {missing}"
    for column_name in columns:
        risk, severity = tags[column_name]
        spark.sql(
            f"ALTER TABLE {TARGET}.{table_name} ALTER COLUMN {column_name} "
            f"SET TAGS ('ig_risk' = '{risk}', 'ig_severity' = '{severity}')"
        )

In [0]:
# Schema information view: the only non-ct object retained.
spark.sql(f"""
CREATE OR REPLACE VIEW {TARGET}.schema AS
SELECT
  c.table_name,
  c.column_name,
  c.data_type,
  COALESCE(c.comment, '') AS column_comment,
  MAX(CASE WHEN t.tag_name = 'ig_risk' THEN t.tag_value END) AS ig_risk,
  MAX(CASE WHEN t.tag_name = 'ig_severity' THEN t.tag_value END) AS ig_severity
FROM {TARGET_CATALOG}.information_schema.columns c
LEFT JOIN {TARGET_CATALOG}.information_schema.column_tags t
  ON t.schema_name = c.table_schema
 AND t.table_name = c.table_name
 AND t.column_name = c.column_name
WHERE c.table_catalog = '{TARGET_CATALOG}'
  AND c.table_schema = '{project_identifier}'
  AND c.table_name IN ('ct_report_tre', 'ct_report_qmul')
GROUP BY c.table_name, c.ordinal_position, c.column_name, c.data_type, c.comment
ORDER BY c.table_name, c.ordinal_position
""")

In [0]:
# Short post-build check. This creates no extra database objects.
summary = spark.sql(f"""
SELECT
  COUNT(*) AS n,
  COUNT(DISTINCT EVENT_ID) AS events,
  COUNT(DISTINCT ACCESSION_NBR) AS accessions,
  COUNT(DISTINCT PSEUDO_ID) AS pseudonyms,
  ROUND(100.0 * COUNT_IF(STUDY_INSTANCE_UID IS NOT NULL) / COUNT(*), 2) AS study_uid_pct,
  ROUND(100.0 * COUNT_IF(REQUEST_DT_TM IS NOT NULL) / COUNT(*), 2) AS request_date_pct
FROM {TARGET}.ct_report_tre
""").collect()[0]

leakage = spark.sql(f"""
WITH source AS (
  SELECT
    lower(REPORT_TEXT_ANON) AS txt,
    lower(trim(MRN)) AS mrn,
    regexp_replace(COALESCE(NHS_NUMBER, ''), '[^0-9]', '') AS nhs,
    lower(ACCESSION_NBR) AS accession,
    person.birth_date
  FROM {TARGET}.ct_report_tre report
  LEFT JOIN 4_prod.bronze.map_person person
    ON person.person_id = report.PERSON_ID
)
SELECT
  SUM(CASE WHEN mrn IS NOT NULL AND length(mrn) >= 6 AND contains(txt, mrn) THEN 1 ELSE 0 END) AS mrn,
  SUM(CASE WHEN length(nhs) = 10 AND (
      contains(txt, nhs)
      OR contains(txt, concat(substr(nhs, 1, 3), ' ', substr(nhs, 4, 3), ' ', substr(nhs, 7, 4)))
      OR contains(txt, concat(substr(nhs, 1, 3), '-', substr(nhs, 4, 3), '-', substr(nhs, 7, 4)))
    ) THEN 1 ELSE 0 END) AS nhs,
  SUM(CASE WHEN accession IS NOT NULL AND contains(txt, accession) THEN 1 ELSE 0 END) AS accession,
  SUM(CASE WHEN birth_date IS NOT NULL AND (
      contains(txt, lower(date_format(birth_date, 'dd/MM/yyyy')))
      OR contains(txt, lower(date_format(birth_date, 'dd.MM.yyyy')))
      OR contains(txt, lower(date_format(birth_date, 'dd-MM-yyyy')))
      OR contains(txt, lower(date_format(birth_date, 'yyyy-MM-dd')))
      OR contains(txt, lower(date_format(birth_date, 'd MMM yyyy')))
    ) THEN 1 ELSE 0 END) AS dob
FROM source
""").collect()[0]

qmul_rows = spark.table(f"{TARGET}.ct_report_qmul").count()
assert summary["n"] == TARGET_N
assert summary["events"] == TARGET_N
assert summary["accessions"] == TARGET_N
assert summary["pseudonyms"] == TARGET_N
assert qmul_rows == TARGET_N

print(
    f"Created {TARGET}.ct_report_tre and {TARGET}.ct_report_qmul: "
    f"{summary['n']} rows each, STUDY_INSTANCE_UID {summary['study_uid_pct']}%, "
    f"Request Date {summary['request_date_pct']}%."
)
print(
    f"Approved anon_text residual check: MRN={leakage['mrn']}, NHS={leakage['nhs']}, "
    f"accession={leakage['accession']}, DOB={leakage['dob']}."
)
if sum((leakage[name] or 0) for name in ("mrn", "nhs", "accession", "dob")):
    print(
        "WARNING: QMUL is not cleared for release. Resolve this through the approved "
        "anonymisation lane/IG process; do not add a project-only scrubber."
    )
